# Exercise 3 (Instructor Solution) — End-to-End Training & Testing Pipeline

Complete reference solution for Exercise 3, including the threshold-tuning bonus task.


In [ ]:
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cust = pd.read_csv("data/customers_raw.csv")
with open("data/usage_events.json", "r", encoding="utf-8") as f:
    usage = pd.DataFrame(json.load(f))
cust["plan_tier"] = cust["plan_tier"].apply(lambda x: x.strip().lower() if isinstance(x, str) else x)
cust["country"] = cust["country"].apply(lambda x: x.strip().upper() if isinstance(x, str) else x)
df = cust.drop_duplicates(subset=["customer_id"], keep="first").merge(usage, on="customer_id", how="left")
df["charges_per_login"] = df["monthly_charges"] / (df["login_days_30d"] + 1.0)
df["tickets_per_tenure_month"] = df["support_tickets_30d"] / (df["tenure_months"] + 1.0)

NUMERIC_COLS = [
    "age", "tenure_months", "monthly_charges",
    "login_days_30d", "watch_hours_30d", "support_tickets_30d",
    "auto_pay", "charges_per_login", "tickets_per_tenure_month",
]
CATEGORICAL_COLS = ["plan_tier", "country", "payment_method", "last_active_device"]

X = df[NUMERIC_COLS + CATEGORICAL_COLS]
y = df["churned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)


In [ ]:
def create_churn_pipeline(numeric_cols: list[str], categorical_cols: list[str]) -> Pipeline:
    """Return an unfitted sklearn Pipeline(preprocessor -> classifier)."""
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]),
                numeric_cols,
            ),
            (
                "cat",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                    ]
                ),
                categorical_cols,
            ),
        ]
    )
    classifier = RandomForestClassifier(
        n_estimators=120, max_depth=6, min_samples_leaf=4, class_weight="balanced", random_state=42
    )
    return Pipeline(steps=[("preprocessor", preprocessor), ("classifier", classifier)])


def train_and_evaluate(
    pipeline: Pipeline,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
) -> tuple[Pipeline, dict]:
    """Run 5-fold CV, fit on X_train, score on X_test, return (pipeline, metrics)."""
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_roc_auc = float(cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="roc_auc").mean())

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    metrics = {
        "cv_roc_auc": round(cv_roc_auc, 4),
        "test_roc_auc": round(float(roc_auc_score(y_test, y_prob)), 4),
        "test_f1": round(float(f1_score(y_test, y_pred)), 4),
    }
    return pipeline, metrics


In [ ]:
pipe = create_churn_pipeline(NUMERIC_COLS, CATEGORICAL_COLS)
fitted_pipe, metrics = train_and_evaluate(pipe, X_train, y_train, X_test, y_test)
print("Evaluation metrics:", metrics)

save_path = Path("artifacts/ex3_pipeline.joblib")
save_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(fitted_pipe, save_path)
reloaded_pipe = joblib.load(save_path)

assert isinstance(fitted_pipe, Pipeline)
assert {"cv_roc_auc", "test_roc_auc", "test_f1"}.issubset(metrics.keys())
assert metrics["cv_roc_auc"] >= 0.72
assert metrics["test_roc_auc"] >= 0.72
assert save_path.exists()

orig_probs = fitted_pipe.predict_proba(X_test)[:, 1]
reloaded_probs = reloaded_pipe.predict_proba(X_test)[:, 1]
assert np.allclose(orig_probs, reloaded_probs)
print("[ALL TESTS PASSED] Pipeline trained, validated, saved, and reloaded identically!")

# Bonus: Threshold tradeoff inspection
print("\nBonus — Threshold Tradeoff on Held-Out Test Set:")
for thresh in [0.35, 0.50, 0.65]:
    preds_t = (orig_probs >= thresh).astype(int)
    p = precision_score(y_test, preds_t, zero_division=0)
    r = recall_score(y_test, preds_t, zero_division=0)
    f = f1_score(y_test, preds_t, zero_division=0)
    print(f"  threshold={thresh:.2f} -> precision={p:.3f} | recall={r:.3f} | f1={f:.3f}")
